In [ ]:
#Importing the required Libraries
import numpy as np
import tensorflow as tf
#Define a function to calculate the numerical derivative
#at the given points x over the given functional values
#using the central difference method
def central_difference(x, y):
  n = len(x)
  dydx = np.zeros(n)
  for i in range(1, n-1):
    dydx[i] = (y[i+1]- y[i-1]) / (x[i+1]- x[i-1])
 # Forward difference at the first point
  dydx[0] = (y[1]- y[0]) / (x[1]- x[0])
 # Backward difference at the last point
  dydx[n-1] = (y[n-1]- y[n-2]) / (x[n-1]- x[n-2])
  return dydx
 # Define some numerical data
x = np.array([-3,-2,-1,0,1,2,3])
y = np.exp(x)
exact_dydx = np.exp(x)
dy_dx = central_difference(x,y)
# Print the result
print(f'{"Numerical Diff Values:":>25} {[f"{val:.6f}" for
val in dy_dx]}\n')
print(f'{"Exact Diff Values:":>25} {[f"{val:.6f}" for val
in exact_dydx]}\n')
print(f'{"The Error":>25} {[f"{val:.6f}" for val in np.
abs(exact_dydx-dy_dx)]}\n')


#Automtic Differentiation
x=tf.Variable(x,dtype=tf.float32)
with tf.GradientTape() as tape:
      y=tf.exp(x)
      dy=tape.gradient(y,x)
# Print the result
print(f'{"Automatic Diff Values:":>25} {[f"{val:.6f}" for
val in dy.numpy()]}\n')
print(f'{"Exact Diff Values:":>25} {[f"{val:.6f}" for val
in exact_dydx]}\n')
print(f'{"The Error":>25} {[f"{val:.6f}" for val in np.
abs(exact_dydx-dy)]}\n')

In [ ]:
import tensorflow as tf
import numpy as np

# ============================================================
# Step 01 – Generate Training Data
# ============================================================

# Example domain: t in [0,1], x in [0,1]
# Modify according to your PDE
def generate_training_data(Nr=2000, N0=200, Nb=200):
    # Residual points (inside domain)
    t_r = np.random.uniform(0, 1, (Nr, 1))
    x_r = np.random.uniform(0, 1, (Nr, 1))

    # Initial condition points t=0
    t_0 = np.zeros((N0, 1))
    x_0 = np.random.uniform(0, 1, (N0, 1))

    # Boundary points x=0 and x=1
    t_b = np.random.uniform(0, 1, (Nb, 1))
    x_b0 = np.zeros((Nb, 1))
    x_b1 = np.ones((Nb, 1))

    return (t_r, x_r), (t_0, x_0), (t_b, x_b0, x_b1)


# ============================================================
# Step 02 – PINN Neural Network Architecture
# ============================================================

class PINN(tf.keras.Model):
    def __init__(self, layers):
        super(PINN, self).__init__()
        self.hidden_layers = []
        for width in layers[:-1]:
            self.hidden_layers.append(tf.keras.layers.Dense(
                width, activation='tanh',
                kernel_initializer='glorot_normal'
            ))
        self.out_layer = tf.keras.layers.Dense(layers[-1])

    def call(self, X):
        Z = X
        for layer in self.hidden_layers:
            Z = layer(Z)
        return self.out_layer(Z)


# ============================================================
# Step 03 – PDE Residual (Modify this for your PDE!)
# ============================================================

def pde_residual(model, t, x):
    # Cast inputs to float32 to ensure consistent dtype
    t_f32 = tf.cast(t, tf.float32)
    x_f32 = tf.cast(x, tf.float32)
    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch([t_f32, x_f32])
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch([t_f32, x_f32])
            inputs = tf.concat([t_f32, x_f32], axis=1)
            u = model(inputs)

        # First-order derivatives
        u_t = tape1.gradient(u, t_f32)
        u_x = tape1.gradient(u, x_f32)

    # Second-order derivative
    u_xx = tape2.gradient(u_x, x_f32)
    del tape1, tape2

    # Example PDE: u_t = u_xx  (Heat equation)
    # Replace this with your F(x, u, derivatives...)
    f = u_t - u_xx
    return f


# ============================================================
# Step 03 – Loss Function
# ============================================================

def loss_fn(model, data):
    (t_r, x_r), (t_0, x_0), (t_b, x_b0, x_b1) = data

    # Residual loss
    # pde_residual now handles casting of t_r and x_r internally
    f_r = pde_residual(model, t_r, x_r)
    loss_r = tf.reduce_mean(tf.square(f_r))

    # Initial condition u(x,0) = g(x)
    # Ensure all components for u_0 and g are float32
    t_0_f32 = tf.cast(t_0, tf.float32)
    x_0_f32 = tf.cast(x_0, tf.float32)
    # Explicitly cast np.pi to float32 as well
    g = tf.sin(tf.constant(np.pi, dtype=tf.float32) * x_0_f32)
    u_0 = model(tf.concat([t_0_f32, x_0_f32], axis=1))
    loss_0 = tf.reduce_mean(tf.square(u_0 - g))

    # Boundary conditions u(0,t) = 0, u(1,t) = 0
    # Ensure all components are float32
    t_b_f32 = tf.cast(t_b, tf.float32)
    x_b0_f32 = tf.cast(x_b0, tf.float32)
    x_b1_f32 = tf.cast(x_b1, tf.float32)
    u_b0 = model(tf.concat([t_b_f32, x_b0_f32], axis=1))
    u_b1 = model(tf.concat([t_b_f32, x_b1_f32], axis=1))
    loss_b = tf.reduce_mean(tf.square(u_b0)) + tf.reduce_mean(tf.square(u_b1))

    return loss_r + loss_0 + loss_b


# ============================================================
# Step 04 – Training Step
# ============================================================

@tf.function
def train_step(model, optimizer, data):
    with tf.GradientTape() as tape:
        loss = loss_fn(model, data)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss


# ============================================================
# Main Training Loop
# ============================================================

def train_PINN():
    # Create model
    model = PINN(layers=[2, 50, 50, 50, 1])

    # Training data
    data = generate_training_data()

    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

    # Training
    for epoch in range(5000):
        loss = train_step(model, optimizer, data)

        if epoch % 500 == 0:
            print(f"Epoch {epoch} | Loss = {loss.numpy():.6f}")

    print("Training complete.")
    return model


# ============================================================
# Run Training
# ============================================================

if __name__ == "__main__":
    model = train_PINN()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# ---------------------------------------------------------
# 1. Create test points
# ---------------------------------------------------------

# Choose a fixed time slice for comparison
t_test_val = 0.5                     # Change this if needed
N_test = 200

x_test = np.linspace(0, 1, N_test).reshape(-1, 1)
t_test = np.full_like(x_test, t_test_val)

# PINN prediction
u_pred = model.predict(np.hstack((t_test, x_test)))  # model from training

# ---------------------------------------------------------
# 2. Exact solution for the example PDE
#    Heat equation u_t = u_xx with IC u(x,0)=sin(pi x)
# ---------------------------------------------------------

u_exact = np.exp(-np.pi**2 * t_test_val) * np.sin(np.pi * x_test)

# ---------------------------------------------------------
# 3. Plot PINN vs Exact
# ---------------------------------------------------------
plt.figure(figsize=(8,5))
plt.plot(x_test, u_exact, label="Exact Solution", linewidth=3)
plt.plot(x_test, u_pred, '--', label="PINN Prediction", linewidth=3)
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.title(f"PINN vs Exact at t = {t_test_val}")
plt.legend()
plt.grid(True)
plt.show()

# ---------------------------------------------------------
# 4. Plot absolute error
# ---------------------------------------------------------
plt.figure(figsize=(8,5))
plt.plot(x_test, np.abs(u_exact - u_pred), label="Error", linewidth=3)
plt.xlabel("x")
plt.ylabel("|Exact - PINN|")
plt.title(f"Absolute Error at t = {t_test_val}")
plt.grid(True)
plt.show()


In [ ]:
# PINN for 2D incompressible Navier-Stokes (Taylor-Green vortex)
# TensorFlow 2.x required.

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from time import time

tf.random.set_seed(0)
np.random.seed(0)

# -----------------------------
# Problem parameters
# -----------------------------
nu = 0.01                     # kinematic viscosity
T_final = 0.5                 # final time
Lx = 2.0 * np.pi
Ly = 2.0 * np.pi

# Taylor-Green analytic solution (domain [0,2pi]x[0,2pi])
def taylor_green_exact(t, x, y, nu=nu):
    """Return u, v, p for Taylor-Green vortex."""
    # ensure arrays
    t = np.array(t).reshape(-1, 1)
    x = np.array(x).reshape(-1, 1)
    y = np.array(y).reshape(-1, 1)
    exp1 = np.exp(-2.0 * nu * t)
    exp2 = np.exp(-4.0 * nu * t)
    u = -np.cos(x) * np.sin(y) * exp1
    v =  np.sin(x) * np.cos(y) * exp1
    p = -0.25 * (np.cos(2*x) + np.cos(2*y)) * exp2
    return u, v, p

# -----------------------------
# Training data generation
# -----------------------------
def latin_hypercube(n, lb, ub):
    """Simple LHS in n-D between lb (array) and ub (array)."""
    dim = len(lb)
    rng = np.random.rand(n, dim)
    result = np.zeros_like(rng)
    for i in range(dim):
        perm = np.random.permutation(n)
        result[:, i] = (perm + rng[:, i]) / n
    return lb + result * (ub - lb)

def make_training_data(Nr=10000, N0=2000, Nb=2000):
    # Residual points in domain (t,x,y)
    Xr = latin_hypercube(Nr, np.array([0.0, 0.0, 0.0]), np.array([T_final, Lx, Ly]))
    t_r = Xr[:, 0:1].astype(np.float32)
    x_r = Xr[:, 1:2].astype(np.float32)
    y_r = Xr[:, 2:3].astype(np.float32)

    # Initial condition points (t=0)
    X0 = latin_hypercube(N0, np.array([0.0, 0.0, 0.0]), np.array([0.0, Lx, Ly]))
    t0 = X0[:, 0:1].astype(np.float32)
    x0 = X0[:, 1:2].astype(np.float32)
    y0 = X0[:, 2:3].astype(np.float32)
    u0, v0, p0 = taylor_green_exact(np.zeros((N0,1)), x0, y0)

    # Boundary matching points for periodic BCs:
    # We'll sample points on x=0 and x=Lx and penalize difference u(.,0,y)-u(.,Lx,y)
    # and similarly for y boundaries.
    Nt = Nb // 2
    tb = np.random.uniform(0.0, T_final, (Nt,1)).astype(np.float32)
    yb = np.random.uniform(0.0, Ly, (Nt,1)).astype(np.float32)
    xb = np.zeros_like(yb)                 # x=0
    xbL = np.full_like(yb, Lx)            # x=Lx

    xb_pts = np.hstack([tb, xb, yb]).astype(np.float32)
    xbL_pts = np.hstack([tb, xbL, yb]).astype(np.float32)

    tb2 = np.random.uniform(0.0, T_final, (Nt,1)).astype(np.float32)
    xb2 = np.random.uniform(0.0, Lx, (Nt,1)).astype(np.float32)
    yb0 = np.zeros_like(xb2)
    ybL = np.full_like(xb2, Ly)

    yb_pts = np.hstack([tb2, xb2, yb0]).astype(np.float32)
    ybL_pts = np.hstack([tb2, xb2, ybL]).astype(np.float32)

    boundary_pairs = {
        'x': (xb_pts, xbL_pts),
        'y': (yb_pts, ybL_pts)
    }

    data = {
        't_r': t_r, 'x_r': x_r, 'y_r': y_r,
        't0': t0, 'x0': x0, 'y0': y0, 'u0': u0.astype(np.float32), 'v0': v0.astype(np.float32),
        'p0': p0.astype(np.float32),
        'boundary_pairs': boundary_pairs
    }
    return data

# -----------------------------
# PINN model: outputs u,v,p
# -----------------------------
class PINN_NS(tf.keras.Model):
    def __init__(self, layers):
        super(PINN_NS, self).__init__()
        self.layers_list = []
        for width in layers[:-1]:
            self.layers_list.append(tf.keras.layers.Dense(width, activation=tf.nn.tanh,
                                                         kernel_initializer='glorot_normal'))
        self.out = tf.keras.layers.Dense(layers[-1])   # outputs 3 values: u,v,p

    def call(self, X):
        Z = X
        for layer in self.layers_list:
            Z = layer(Z)
        return self.out(Z)

# -----------------------------
# PDE residual function
# -----------------------------
# Uses nested GradientTape to compute derivatives w.r.t t,x,y
def pde_residuals(model, t, x, y):
    # inputs should be tf tensors of shape (N,1)
    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch([t, x, y])
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch([t, x, y])
            X = tf.concat([t, x, y], axis=1)
            uvp = model(X)                          # (N,3)
            u = uvp[:, 0:1]
            v = uvp[:, 1:2]
            p = uvp[:, 2:3]
        # first derivatives
        u_t = tape1.gradient(u, t)
        u_x = tape1.gradient(u, x)
        u_y = tape1.gradient(u, y)

        v_t = tape1.gradient(v, t)
        v_x = tape1.gradient(v, x)
        v_y = tape1.gradient(v, y)

        p_x = tape1.gradient(p, x)
        p_y = tape1.gradient(p, y)

    # second derivatives for diffusion term
    u_xx = tape2.gradient(u_x, x)
    u_yy = tape2.gradient(u_y, y)

    v_xx = tape2.gradient(v_x, x)
    v_yy = tape2.gradient(v_y, y)

    # continuity
    div = u_x + v_y

    # Navier-Stokes momentum residuals (incompressible)
    # u_t + u*u_x + v*u_y = -p_x + nu*(u_xx + u_yy)
    # v_t + u*v_x + v*v_y = -p_y + nu*(v_xx + v_yy)
    # All in strong form
    momentum_u = u_t + (u * u_x) + (v * u_y) + p_x - nu * (u_xx + u_yy)
    momentum_v = v_t + (u * v_x) + (v * v_y) + p_y - nu * (v_xx + v_yy)

    # cleanup tapes
    del tape1, tape2

    return momentum_u, momentum_v, div, u, v, p

# -----------------------------
# Loss function
# -----------------------------
def loss_fn(model, data):
    # Residual loss
    t_r = tf.convert_to_tensor(data['t_r'])
    x_r = tf.convert_to_tensor(data['x_r'])
    y_r = tf.convert_to_tensor(data['y_r'])

    m_u, m_v, div, _, _, _ = pde_residuals(model, t_r, x_r, y_r)
    loss_r = tf.reduce_mean(tf.square(m_u)) + tf.reduce_mean(tf.square(m_v)) + tf.reduce_mean(tf.square(div))

    # Initial condition loss
    t0 = tf.convert_to_tensor(data['t0'])
    x0 = tf.convert_to_tensor(data['x0'])
    y0 = tf.convert_to_tensor(data['y0'])
    uvp0 = model(tf.concat([t0, x0, y0], axis=1))
    u0_pred = uvp0[:, 0:1]
    v0_pred = uvp0[:, 1:2]
    p0_pred = uvp0[:, 2:3]

    u0 = tf.convert_to_tensor(data['u0'])
    v0 = tf.convert_to_tensor(data['v0'])
    p0 = tf.convert_to_tensor(data['p0'])

    loss_ic = tf.reduce_mean(tf.square(u0_pred - u0)) + tf.reduce_mean(tf.square(v0_pred - v0)) \
              + tf.reduce_mean(tf.square(p0_pred - p0))

    # Boundary periodic matching loss (weak enforcement)
    bp = data['boundary_pairs']
    loss_b = 0.0
    # x boundaries: enforce equality between x=0 and x=Lx
    xb, xbL = bp['x']
    xb = tf.convert_to_tensor(xb)
    xbL = tf.convert_to_tensor(xbL)
    uvp_x0 = model(xb)
    uvp_xL = model(xbL)
    loss_b += tf.reduce_mean(tf.square(uvp_x0 - uvp_xL))

    # y boundaries
    yb, ybL = bp['y']
    yb = tf.convert_to_tensor(yb)
    ybL = tf.convert_to_tensor(ybL)
    uvp_y0 = model(yb)
    uvp_yL = model(ybL)
    loss_b += tf.reduce_mean(tf.square(uvp_y0 - uvp_yL))

    # Total loss
    LOSS = loss_r + 100.0 * loss_ic + 10.0 * loss_b   # weights: IC more strongly enforced
    return LOSS, loss_r, loss_ic, loss_b

# -----------------------------
# Training step + loop
# -----------------------------
@tf.function
def train_step(model, optimizer, data):
    with tf.GradientTape() as tape:
        LOSS, loss_r, loss_ic, loss_b = loss_fn(model, data)
    grads = tape.gradient(LOSS, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return LOSS, loss_r, loss_ic, loss_b

def train(model, data, epochs=30000, lr=1e-3, print_every=500):
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    history = []
    t0 = time()
    for ep in range(1, epochs+1):
        LOSS, loss_r, loss_ic, loss_b = train_step(model, optimizer, data)
        if ep % print_every == 0 or ep == 1:
            elapsed = time() - t0
            print(f"Epoch {ep:6d} | LOSS {LOSS.numpy():.6e} | res {loss_r.numpy():.3e} | ic {loss_ic.numpy():.3e} | b {loss_b.numpy():.3e} | time {elapsed:.1f}s")
            history.append((ep, LOSS.numpy(), loss_r.numpy(), loss_ic.numpy(), loss_b.numpy()))
    return history

# -----------------------------
# Build data, model, train
# -----------------------------
data = make_training_data(Nr=8000, N0=2000, Nb=2000)

model = PINN_NS(layers=[3, 128, 128, 128, 128, 3])   # 3 inputs -> hidden -> 3 outputs (u,v,p)

# initialize with a small forward pass (build weights)
_ = model(tf.random.uniform((10,3)))

# Train — adjust epochs if you have more compute
hist = train(model, data, epochs=8000, lr=1e-3, print_every=500)

# -----------------------------
# Evaluate & plot PINN vs exact at t = t_plot
# -----------------------------
def evaluate_on_grid(model, t_val, Nx=120, Ny=120):
    xs = np.linspace(0, Lx, Nx)
    ys = np.linspace(0, Ly, Ny)
    X, Y = np.meshgrid(xs, ys)
    TT = np.full_like(X, t_val)
    pts = np.vstack([TT.ravel(), X.ravel(), Y.ravel()]).T.astype(np.float32)
    uvp_pred = model(pts).numpy()
    u_pred = uvp_pred[:,0].reshape(Ny, Nx)
    v_pred = uvp_pred[:,1].reshape(Ny, Nx)
    p_pred = uvp_pred[:,2].reshape(Ny, Nx)
    u_ex, v_ex, p_ex = taylor_green_exact(t_val, X.ravel()[:,None], Y.ravel()[:,None])
    u_ex = u_ex.reshape(Ny, Nx)
    v_ex = v_ex.reshape(Ny, Nx)
    p_ex = p_ex.reshape(Ny, Nx)
    return X, Y, u_pred, v_pred, p_pred, u_ex, v_ex, p_ex

t_plot = 0.2
X, Y, u_pred, v_pred, p_pred, u_ex, v_ex, p_ex = evaluate_on_grid(model, t_plot, Nx=100, Ny=100)

# Plot u field: exact vs pred vs abs error
fig, axs = plt.subplots(1,3, figsize=(18,5))
im0 = axs[0].contourf(X, Y, u_ex, 50); axs[0].set_title("u_exact")
im1 = axs[1].contourf(X, Y, u_pred, 50); axs[1].set_title("u_pred (PINN)")
im2 = axs[2].contourf(X, Y, np.abs(u_ex-u_pred), 50); axs[2].set_title("abs error (u)")
for a in axs:
    a.set_xlabel("x"); a.set_ylabel("y")
fig.colorbar(im2, ax=axs.ravel().tolist(), shrink=0.6)
plt.tight_layout()
plt.show()

# Plot v field
fig, axs = plt.subplots(1,3, figsize=(18,5))
im0 = axs[0].contourf(X, Y, v_ex, 50); axs[0].set_title("v_exact")
im1 = axs[1].contourf(X, Y, v_pred, 50); axs[1].set_title("v_pred (PINN)")
im2 = axs[2].contourf(X, Y, np.abs(v_ex-v_pred), 50); axs[2].set_title("abs error (v)")
for a in axs:
    a.set_xlabel("x"); a.set_ylabel("y")
fig.colorbar(im2, ax=axs.ravel().tolist(), shrink=0.6)
plt.tight_layout()
plt.show()

# Print L2 error norms
err_u = np.linalg.norm((u_ex - u_pred).ravel(), 2) / np.linalg.norm(u_ex.ravel(), 2)
err_v = np.linalg.norm((v_ex - v_pred).ravel(), 2) / np.linalg.norm(v_ex.ravel(), 2)
err_p = np.linalg.norm((p_ex - p_pred).ravel(), 2) / np.linalg.norm(p_ex.ravel(), 2)
print(f"Relative L2 errors at t={t_plot}: u: {err_u:.3e}, v: {err_v:.3e}, p: {err_p:.3e}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# ============================================================
# PINN Neural Network Architecture (Copied from training cell)
# ============================================================
class PINN(tf.keras.Model):
    def __init__(self, layers):
        super(PINN, self).__init__()
        self.hidden_layers = []
        for width in layers[:-1]:
            self.hidden_layers.append(tf.keras.layers.Dense(
                width, activation='tanh',
                kernel_initializer='glorot_normal'
            ))
        self.out_layer = tf.keras.layers.Dense(layers[-1])

    def call(self, X):
        Z = X
        for layer in self.hidden_layers:
            Z = layer(Z)
        return self.out_layer(Z)

# ============================================================
# PDE Residual (Copied from training cell)
# ============================================================
def pde_residual(model, t, x):
    t_f32 = tf.cast(t, tf.float32)
    x_f32 = tf.cast(x, tf.float32)
    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch([t_f32, x_f32])
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch([t_f32, x_f32])
            inputs = tf.concat([t_f32, x_f32], axis=1)
            u = model(inputs)
        u_t = tape1.gradient(u, t_f32)
        u_x = tape1.gradient(u, x_f32)
    u_xx = tape2.gradient(u_x, x_f32)
    del tape1, tape2
    f = u_t - u_xx
    return f

# ============================================================
# Loss Function (Copied from training cell)
# ============================================================
def loss_fn(model, data):
    (t_r, x_r), (t_0, x_0), (t_b, x_b0, x_b1) = data
    f_r = pde_residual(model, t_r, x_r)
    loss_r = tf.reduce_mean(tf.square(f_r))

    t_0_f32 = tf.cast(t_0, tf.float32)
    x_0_f32 = tf.cast(x_0, tf.float32)
    g = tf.sin(tf.constant(np.pi, dtype=tf.float32) * x_0_f32)
    u_0 = model(tf.concat([t_0_f32, x_0_f32], axis=1))
    loss_0 = tf.reduce_mean(tf.square(u_0 - g))

    t_b_f32 = tf.cast(t_b, tf.float32)
    x_b0_f32 = tf.cast(x_b0, tf.float32)
    x_b1_f32 = tf.cast(x_b1, tf.float32)
    u_b0 = model(tf.concat([t_b_f32, x_b0_f32], axis=1))
    u_b1 = model(tf.concat([t_b_f32, x_b1_f32], axis=1))
    loss_b = tf.reduce_mean(tf.square(u_b0)) + tf.reduce_mean(tf.square(u_b1))

    return loss_r + loss_0 + loss_b

# ============================================================
# Training Step (Copied from training cell)
# ============================================================
@tf.function
def train_step(model, optimizer, data):
    with tf.GradientTape() as tape:
        loss = loss_fn(model, data)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

# ============================================================
# Generate Training Data (Copied from training cell)
# ============================================================
def generate_training_data(Nr=2000, N0=200, Nb=200):
    t_r = np.random.uniform(0, 1, (Nr, 1))
    x_r = np.random.uniform(0, 1, (Nr, 1))
    t_0 = np.zeros((N0, 1))
    x_0 = np.random.uniform(0, 1, (N0, 1))
    t_b = np.random.uniform(0, 1, (Nb, 1))
    x_b0 = np.zeros((Nb, 1))
    x_b1 = np.ones((Nb, 1))
    return (t_r, x_r), (t_0, x_0), (t_b, x_b0, x_b1)

# ============================================================
# Main Training Function (Copied from training cell)
# ============================================================
def train_PINN():
    model = PINN(layers=[2, 50, 50, 50, 1])
    data = generate_training_data()
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

    for epoch in range(5000):
        loss = train_step(model, optimizer, data)
        if epoch % 500 == 0:
            print(f"Epoch {epoch} | Loss = {loss.numpy():.6f}")
    print("Training complete.")
    return model

# ============================================================
# Train the model (re-run training to get the model variable)
# ============================================================
print("Re-training PINN model for plotting...")
model = train_PINN()

# ---------------------------------------------------------
# 1. Create test points
# ---------------------------------------------------------

t_test_val = 0.5                     # Change this if needed
N_test = 200

x_test = np.linspace(0, 1, N_test).reshape(-1, 1)
t_test = np.full_like(x_test, t_test_val)

# PINN prediction
u_pred = model.predict(np.hstack((t_test, x_test)))

# ---------------------------------------------------------
# 2. Exact solution for the example PDE
# ---------------------------------------------------------

u_exact = np.exp(-np.pi**2 * t_test_val) * np.sin(np.pi * x_test)

# ---------------------------------------------------------
# 3. Plot PINN vs Exact
# ---------------------------------------------------------
plt.figure(figsize=(8,5))
plt.plot(x_test, u_exact, label="Exact Solution", linewidth=3)
plt.plot(x_test, u_pred, '--', label="PINN Prediction", linewidth=3)
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.title(f"PINN vs Exact at t = {t_test_val}")
plt.legend()
plt.grid(True)
plt.show()